<a href="https://colab.research.google.com/github/Ashish131989/NLPExploration/blob/main/RAG_Pincone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
documents = [
    {
        "id": "health_001",
        "text": """
        Health insurance provides financial coverage for medical expenses.
        The policy covers hospitalization expenses, doctor consultation,
        surgery, diagnostic tests, and ambulance charges.

        Pre-existing diseases are covered after a waiting period of
        24 months from the policy start date.

        Cashless treatment is available at network hospitals.
        """
    },

    {
        "id": "health_002",
        "text": """
        To file a health insurance claim, the policyholder must inform
        the insurance company within 24 hours of emergency hospitalization.

        Required documents include hospital bills, discharge summary,
        medical reports, and policy details.
        """
    },

    {
        "id": "auto_001",
        "text": """
        Motor insurance covers damages caused to the insured vehicle.

        Comprehensive insurance covers accidents, theft, fire,
        and natural disasters.

        Third-party insurance covers damages caused to another person
        or their property.
        """
    },

    {
        "id": "auto_002",
        "text": """
        Motor insurance does not cover normal wear and tear,
        mechanical breakdown, or damage caused while driving
        under the influence of alcohol.
        """
    }
]

In [2]:
!pip install pinecone sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 18.0 MB/s eta 0:00:00


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
text = "What is the waiting period?"

embedding = model.encode(text)

print(len(embedding))

384


In [5]:
def chunk_text(text, chunk_size=200, overlap=50):

    words = text.split()

    chunks = []

    for i in range(
        0,
        len(words),
        chunk_size - overlap
    ):

        chunk = words[i:i + chunk_size]

        chunks.append(
            " ".join(chunk)
        )

    return chunks

In [6]:
all_chunks = []

for doc in documents:

    chunks = chunk_text(
        doc["text"]
    )

    for i, chunk in enumerate(chunks):

        all_chunks.append({
            "id": f"{doc['id']}_chunk_{i}",
            "text": chunk
        })

In [7]:
for chunk in all_chunks:
    print(chunk)

{'id': 'health_001_chunk_0', 'text': 'Health insurance provides financial coverage for medical expenses. The policy covers hospitalization expenses, doctor consultation, surgery, diagnostic tests, and ambulance charges. Pre-existing diseases are covered after a waiting period of 24 months from the policy start date. Cashless treatment is available at network hospitals.'}
{'id': 'health_002_chunk_0', 'text': 'To file a health insurance claim, the policyholder must inform the insurance company within 24 hours of emergency hospitalization. Required documents include hospital bills, discharge summary, medical reports, and policy details.'}
{'id': 'auto_001_chunk_0', 'text': 'Motor insurance covers damages caused to the insured vehicle. Comprehensive insurance covers accidents, theft, fire, and natural disasters. Third-party insurance covers damages caused to another person or their property.'}
{'id': 'auto_002_chunk_0', 'text': 'Motor insurance does not cover normal wear and tear, mechanic

In [9]:
texts = [
    chunk["text"]
    for chunk in all_chunks
]

In [11]:
embeddings = model.encode(
    texts
)
print (embeddings)

[[-0.03558844  0.14267732  0.04176487 ... -0.08014353  0.0635303
   0.02999332]
 [-0.01918689  0.13218378  0.03500832 ...  0.03528972 -0.05053852
   0.01408474]
 [-0.06942017  0.02381638  0.09186592 ... -0.02433642  0.05611678
   0.05734525]
 [ 0.00446269  0.07803234  0.09915489 ... -0.02827482  0.07209091
   0.02953025]]


In [38]:
from pinecone import Pinecone
from google.colab import userdata
PINECONE_KEY = userdata.get("PINECONE_KEY")
pc = Pinecone(
    api_key=PINECONE_KEY
)

In [39]:
PINECONE_KEY

'pcsk_6yQVEp_CZuP4nHhaGyXoBy4hpsEd6NmMQHxCUZC7aHqcwgqUiCYNQUVt5SZocvsadnkc9c'

In [13]:
from pinecone import ServerlessSpec

index_name = "insurance-rag"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",

    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

Name:,insurance-rag
Status:,Ready
Ready:,Yes
Deployment:,Managed (aws/us-east-1)
Host:,https://insurance-rag-6ky4whc.svc.aped-4627-b74a.pinecone.io
Deletion Protection:,disabled
Schema fields:,2
Read capacity:,"ReadCapacityOnDemandResponse(status=ReadCapacityStatus(state='Ready', current_shards=None, current_replicas=None, error_message=None))"


In [14]:
index = pc.Index(
    "insurance-rag"
)

In [15]:
vectors = []

for chunk, embedding in zip(
    all_chunks,
    embeddings
):

    vectors.append({

        "id": chunk["id"],

        "values": embedding.tolist(),

        "metadata": {

            "text": chunk["text"],

            "document_id":
                chunk["id"].split("_chunk")[0]
        }
    })

In [16]:
index.upsert(
    vectors=vectors
)

UpsertResponse(upserted_count=4)

In [17]:
query = """
What is the waiting period
for pre-existing diseases?
"""

query_embedding = model.encode(
    query
)

In [18]:
results = index.query(

    vector=query_embedding.tolist(),

    top_k=3,

    include_metadata=True
)

In [23]:
print(results)

QueryResponse(matches=[ScoredVector(id='health_001_chunk_0', score=0.454846382, values=[], metadata={'document_id': 'health_001', 'text': 'Health insurance provides financial coverage for medical expenses. The policy covers hospitalization expenses, doctor consultation, surgery, diagnostic tests, and ambulance charges. Pre-existing diseases are covered after a waiting period of 24 months from the policy start date. Cashless treatment is available at network hospitals.'}), ScoredVector(id='health_002_chunk_0', score=0.252250671, values=[], metadata={'document_id': 'health_002', 'text': 'To file a health insurance claim, the policyholder must inform the insurance company within 24 hours of emergency hospitalization. Required documents include hospital bills, discharge summary, medical reports, and policy details.'}), ScoredVector(id='auto_002_chunk_0', score=0.00931310654, values=[], metadata={'document_id': 'auto_002', 'text': 'Motor insurance does not cover normal wear and tear, mechan

In [24]:
context = "\n\n".join(

    match["metadata"]["text"]

    for match in results["matches"]
)

In [25]:
!pip install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 14.7 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [40]:
import os
from google import genai
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")


In [41]:
prompt = f"""
You are an AI assistant for an insurance company.

Answer the user's question using ONLY the information
provided in the context below.

Rules:

1. Do not use information outside the context.
2. Do not hallucinate or make up information.
3. If the answer is not available in the context, say:
   "I could not find this information in the insurance documents."
4. Keep the answer clear and concise.

CONTEXT:
{context}

USER QUESTION:
{query}
"""

In [42]:
client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

In [43]:
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt
)

In [44]:
print(response.text)

The waiting period for pre-existing diseases is 24 months from the policy start date.
